# Kontrolowane zmiany w kodzie (mutacje)

W tym zadaniu wprowadzamy do testowanych funkcji małe zmiany i sprawdzamy, czy nasze testy czarnoskrzynkowe je wykryją.

Zrobiliśmy 20 zmian w 14 funkcjach:
- zamiana `<` na `<=` albo odwrotnie (też `>` i `>=`): M1, M4, M5, M10, M14, M17, M18,
- zmiana wartości granicznej: M2, M7, M13, M19,
- zamiana `and` na `or` albo odwrotnie: M3, M6, M9,
- zmiana zwracanej wartości: M11, M12, M15, M16,
- usunięcie zgłaszanego wyjątku: M8, M20.

Funkcje wybraliśmy tak, żeby miały warunki i granice, w których łatwo o błąd (długość sluga, paginacja, sprawdzanie ścieżek, rozmiar obrazka, logowanie).

## Jak to zrobiliśmy

Zmiany nie są robione ręcznie w edytorze, tylko przez kod w notebooku. Dla każdej zmiany:
1. zapamiętujemy oryginalny plik,
2. zamieniamy w nim jeden fragment kodu na zmieniony,
3. uruchamiamy cały zestaw testów czarnoskrzynkowych i zapisujemy, które testy nie przeszły,
4. przywracamy oryginalny plik (w `finally`, więc nawet przy błędzie kod wraca do oryginału).

Dzięki temu w kodzie jest zawsze tylko jedna zmiana naraz, a przed następną kod jest już przywrócony.
Zmiana jest **wykryta**, jeśli przynajmniej jeden test nie przeszedł.

In [2]:
from pathlib import Path

ROOT = Path("../..").resolve()

# każda zmiana: w pliku `plik` fragment `przed` zamieniamy na `po` (fragment musi występować w pliku dokładnie raz)
ZMIANY = [
    dict(nr="M1", funkcja="B1 paginate", rodzaj="< na <=", plik="backend/utils.py",
         przed="has_more = end < total", po="has_more = end <= total"),
    dict(nr="M2", funkcja="B1 paginate", rodzaj="wartość graniczna", plik="backend/utils.py",
         przed="offset = max(0, min(offset, total))", po="offset = max(1, min(offset, total))"),
    dict(nr="M3", funkcja="B3 get_media_type", rodzaj="and na or", plik="backend/utils.py",
         przed="if name_lower.startswith('drawing-') and name_lower.endswith('.png'):",
         po="if name_lower.startswith('drawing-') or name_lower.endswith('.png'):"),
    dict(nr="M4", funkcja="B5 parse_image_size_spec", rodzaj="> na >=", plik="backend/export.py",
         przed="trimmed[:idx].strip(),\n        w if w > 0 else None,",
         po="trimmed[:idx].strip(),\n        w if w >= 0 else None,"),
    dict(nr="M5", funkcja="B5 parse_image_size_spec", rodzaj="> na >=", plik="backend/export.py",
         przed='"",\n                w if w > 0 else None,',
         po='"",\n                w if w >= 0 else None,'),
    dict(nr="M6", funkcja="B9 _normalize_share_public_origin", rodzaj="or na and", plik="backend/main.py",
         przed="if parsed.scheme not in ('http', 'https') or not parsed.netloc:",
         po="if parsed.scheme not in ('http', 'https') and not parsed.netloc:"),
    dict(nr="M7", funkcja="B11 calculate_stats", rodzaj="wartość graniczna", plik="plugins/note_stats.py",
         przed="WORDS_PER_MINUTE = 200", po="WORDS_PER_MINUTE = 250"),
    dict(nr="M8", funkcja="B12 MCPConfig", rodzaj="usunięcie wyjątku", plik="mcp_server/config.py",
         przed='raise ValueError(f"URL must use http or https: {self.base_url}")', po="pass"),
    dict(nr="M9", funkcja="B13 _validate_path", rodzaj="or na and", plik="mcp_server/server.py",
         przed='if path.startswith("/") or path.startswith("\\\\"):',
         po='if path.startswith("/") and path.startswith("\\\\"):'),
    dict(nr="M10", funkcja="B13 _validate_path", rodzaj=">= na >", plik="mcp_server/server.py",
         przed='if len(path) >= 2 and path[1] == ":":', po='if len(path) > 2 and path[1] == ":":'),
    dict(nr="M11", funkcja="B15 get_tool_by_name", rodzaj="zwracana wartość", plik="mcp_server/tools.py",
         przed="            return tool\n    return None", po="            return tool\n    return TOOLS[0]"),
    dict(nr="M12", funkcja="B16 parse_tags", rodzaj="zwracana wartość", plik="backend/utils.py",
         przed="return sorted(list(set(tags)))", po="return sorted(tags)"),
    dict(nr="M13", funkcja="B20 extract_search_terms", rodzaj="wartość graniczna", plik="backend/note_index.py",
         przed='_SEARCH_TOKEN_RE = re.compile(r"[A-Za-z0-9_\\-]{2,}")',
         po='_SEARCH_TOKEN_RE = re.compile(r"[A-Za-z0-9_\\-]{3,}")'),
    dict(nr="M14", funkcja="B21 parse_theme_metadata", rodzaj="> na >=", plik="backend/themes.py",
         przed="if i > 10:", po="if i >= 10:"),
    dict(nr="M15", funkcja="B22 verify_api_key", rodzaj="zwracana wartość", plik="backend/main.py",
         przed="configured_key.encode('utf-8'))\n    except Exception:\n        return False",
         po="configured_key.encode('utf-8'))\n    except Exception:\n        return True"),
    dict(nr="M16", funkcja="B23 verify_password", rodzaj="zwracana wartość", plik="backend/main.py",
         przed='logger.error("Password verification error: %s", e)\n        return False',
         po='logger.error("Password verification error: %s", e)\n        return True'),
    dict(nr="M17", funkcja="B24 _open_tasks", rodzaj="> na >=", plik="plugins/contrib/search_open_tasks.py",
         przed="if len(text) > MAX_TASK_CHARS:", po="if len(text) >= MAX_TASK_CHARS:"),
    dict(nr="M18", funkcja="B25 validate_slug", rodzaj="< na <=", plik="backend/share.py",
         przed="if len(candidate) < SLUG_MIN_LENGTH:", po="if len(candidate) <= SLUG_MIN_LENGTH:"),
    dict(nr="M19", funkcja="B25 validate_slug", rodzaj="wartość graniczna", plik="backend/share.py",
         przed="SLUG_MAX_LENGTH = 64", po="SLUG_MAX_LENGTH = 65"),
    dict(nr="M20", funkcja="B25 validate_slug", rodzaj="usunięcie wyjątku", plik="backend/share.py",
         przed="raise ShareSlugError('too_long')", po="pass"),
]

for z in ZMIANY:
    tekst = (ROOT / z["plik"]).read_text(encoding="utf-8")
    assert tekst.count(z["przed"]) == 1, z["nr"]
print(len(ZMIANY), "zmian, każda pasuje do kodu w jednym miejscu")

20 zmian, każda pasuje do kodu w jednym miejscu


Uruchomienie wszystkich zmian. Na początku sprawdzamy, że bez zmian wszystkie testy przechodzą. Całość trwa kilkanaście sekund.

In [ ]:
import os, subprocess, sys, tempfile
import xml.etree.ElementTree as ET

def uruchom_testy():
    raport = Path(tempfile.gettempdir()) / "mutacje.xml"
    # bez zapisywania plików .pyc, żeby Python zawsze czytał aktualną wersję kodu
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")
    subprocess.run([sys.executable, "-m", "pytest", "tests/unit/blackbox", "-q", "-p", "no:cacheprovider",
                    f"--junitxml={raport}"], cwd=ROOT, env=env, capture_output=True)
    nieudane = []
    for tc in ET.parse(raport).getroot().iter("testcase"):
        if tc.find("failure") is not None or tc.find("error") is not None:
            nieudane.append(tc.get("name"))
    return nieudane


wyniki = {}
for z in ZMIANY:
    plik = ROOT / z["plik"]
    oryginal = plik.read_text(encoding="utf-8")
    try:
        plik.write_text(oryginal.replace(z["przed"], z["po"]), encoding="utf-8")
        wyniki[z["nr"]] = uruchom_testy()
    finally:
        plik.write_text(oryginal, encoding="utf-8")   # przywrócenie oryginału

    nieudane = wyniki[z["nr"]]
    if nieudane:
        opis = f"WYKRYTA, nie przeszło {len(nieudane)}: {nieudane}"
    else:
        opis = "NIE WYKRYTA"
    print(f"{z['nr']:4} {z['funkcja']:34} {z['rodzaj']:18} {opis}")

Bez zmian w kodzie, nieudane testy: []
M1   B1 paginate                        < na <=            WYKRYTA, nie przeszło 1: ['test_paginate_bounds[10-0-expected6-False]']
M2   B1 paginate                        wartość graniczna  WYKRYTA, nie przeszło 4: ['test_paginate_bounds[3-0-expected0-True]', 'test_paginate_bounds[3--5-expected4-True]', 'test_paginate_bounds[10-0-expected6-False]', 'test_paginate_sort_and_to_dict']
M3   B3 get_media_type                  and na or          WYKRYTA, nie przeszło 1: ['test_get_media_type[a.png-image]']
M4   B5 parse_image_size_spec           > na >=            WYKRYTA, nie przeszło 1: ['test_parse_image_size_spec[c|0x200-False-expected6]']
M5   B5 parse_image_size_spec           > na >=            NIE WYKRYTA
M6   B9 _normalize_share_public_origin  or na and          WYKRYTA, nie przeszło 2: ['test_normalize_share_public_origin[ftp://example.com-]', 'test_normalize_share_public_origin[http://-]']
M7   B11 calculate_stats                wartość grani

## Wyniki

| Nr | Funkcja | Zmiana | Wykryta? | Test, który wykrył / powinien wykryć |
|---|---|---|---|---|
| M1 | B1 paginate | `has_more = end < total` -> `<=` | tak | `test_paginate_bounds` z `limit=10` (limit równy długości listy). Wykrył tylko ten jeden przypadek |
| M2 | B1 paginate | `max(0, ...)` -> `max(1, ...)` przy offsecie | tak | `test_paginate_bounds` z `offset=0` i z ujemnym offsetem |
| M3 | B3 get_media_type | `and` -> `or` przy rozpoznawaniu `drawing-*.png` | tak | `test_get_media_type[a.png]`: zwykły PNG zaczął być rozpoznawany jako rysunek |
| M4 | B5 parse_image_size_spec | `w > 0` -> `w >= 0` (zwykły zapis `opis\|rozmiar`) | tak | `test_parse_image_size_spec` z `c\|0x200` |
| M5 | B5 parse_image_size_spec | `w > 0` -> `w >= 0` (sam rozmiar, `allow_solo=True`) | **nie** | powinien: `("0x200", allow_solo=True)` -> `("", None, 200)`. Mamy szerokość 0 tylko w zwykłym zapisie |
| M6 | B9 _normalize_share_public_origin | `or not parsed.netloc` -> `and` | tak | przypadki `ftp://example.com` i `http://` |
| M7 | B11 calculate_stats | 200 słów na minutę -> 250 | **nie** | powinien: `test_stats_reading_time_grows_with_words`, gdyby sprawdzał dokładną wartość (2000 słów = 10 minut według dokumentacji). Teraz sprawdza tylko, że czas rośnie |
| M8 | B12 MCPConfig | usunięty `raise ValueError` dla złego protokołu | tak | `test_config_invalid_url[ftp://host]` |
| M9 | B13 _validate_path | `startswith("/") or startswith("\\")` -> `and` | tak | `test_validate_path_rejects` z `/abs.md` i `\abs.md` |
| M10 | B13 _validate_path | `len(path) >= 2` -> `> 2` przy literze dysku | **nie** | powinien: ścieżka `"C:"` (dokładnie 2 znaki). Nasz przypadek `C:\x.md` jest dłuższy, więc odrzuca go też zmieniony warunek |
| M11 | B15 get_tool_by_name | `return None` -> `return TOOLS[0]` | tak | `test_tool_lookup` (nieznana nazwa) |
| M12 | B16 parse_tags | bez usuwania powtórzeń (`set`) | tak | przypadki z powtórzonymi tagami w `test_parse_tags_extracts` |
| M13 | B20 extract_search_terms | najkrótsze słowo 2 -> 3 znaki | tak | `test_search_terms` z `a b go` i `v2 2026` |
| M14 | B21 parse_theme_metadata | czytanie pierwszych 11 linii -> 10 (`i > 10` -> `i >= 10`) | **nie** | powinien: nagłówek w 11. linii pliku. Takiego testu nie da się napisać ze specyfikacji, patrz niżej |
| M15 | B22 verify_api_key | w `except` `return False` -> `return True` | **nie** | żaden test tego nie wykryje, ten kod jest nieosiągalny (patrz niżej) |
| M16 | B23 verify_password | w `except` `return False` -> `return True` | tak | `test_verify_password_malformed_hash`: uszkodzony hash zaczął przepuszczać każde hasło |
| M17 | B24 _open_tasks | `len(text) > 200` -> `>=` | tak | `test_open_task_long_text_is_truncated[200]` |
| M18 | B25 validate_slug | `len < 3` -> `len <= 3` | tak | 11 przypadków, m.in. `abc` i `ABC` (slug o długości 3) |
| M19 | B25 validate_slug | maksymalna długość 64 -> 65 | tak | `test_slug_rejected` z 65 znakami |
| M20 | B25 validate_slug | usunięty `raise ShareSlugError('too_long')` | tak | `test_slug_rejected` z 65 znakami |

### Niewykryte zmiany

- **M5 i M10** to nasze braki. Oba przypadki dało się wymyślić ze specyfikacji: docstring `parse_image_size_spec` mówi, że wymiar 0 znaczy „nieustawiony”,
  a `C:` to po prostu litera dysku. Sprawdziliśmy tylko jeden wariant, a funkcja ma dwie osobne ścieżki (M5) albo granicę dokładnie przy 2 znakach (M10).
- **M7** pokazuje słaby test. Test przechodzi przez kod liczący czas czytania (pokrycie 100%), ale nie sprawdza wyniku dokładnie.
  Wystarczyłoby `assert ...["reading_time_minutes"] == 10` dla 2000 słów.
- **M14** zmienia szczegół, którego nie ma w dokumentacji. `THEMES.md` mówi tylko, że nagłówek ma być „na początku pliku”, bez liczby linii.
  Nasz test z nagłówkiem po 20 liniach sprawdza tylko, że plik nie jest czytany w całości. Żeby wykryć tę zmianę, trzeba znać limit z kodu, czyli byłby to test białoskrzynkowy.
- **M15** jest w bloku `except` w `verify_api_key`, który według analizy pokrycia jest nieosiągalny dla poprawnych danych.
  Kod w tym miejscu się nie wykonuje, więc żaden test nie może wykryć zmiany. Taką zmianę można traktować jako równoważną oryginałowi.

## Wnioski

- Testy wykryły 15 z 20 zmian (75%). Jeśli nie liczyć M15, której nie da się wykryć żadnym testem, to 15 z 19 (79%).
- Najlepiej działają przypadki na wartościach granicznych. M1, M4, M17, M19 i M20 wykrył tylko jeden przypadek, i zawsze był to przypadek dokładnie na granicy
  (limit równy długości listy, szerokość 0, tekst 200 znaków, slug 65 znaków). Bez nich te zmiany by przeszły.
- Pokrycie kodu nie gwarantuje wykrycia zmian. M5, M7 i M10 są w liniach, które testy wykonują, a mimo to zmiany przeszły.
  Przy M7 linia jest wykonana, ale test nie sprawdza wyniku dokładnie. Przy M5 i M10 brakuje przypadku dokładnie na granicy.
- Jedna zmiana (M15) potwierdza wynik analizy pokrycia: kodu, który się nie wykonuje, nie da się sprawdzić testem.
- Przypadki, które należałoby dodać: `("0x200", allow_solo=True)` w B5, `"C:"` w B13 i dokładny czas czytania w B11.
  Wszystkie da się wymyślić z samej specyfikacji, więc pasują do testów czarnoskrzynkowych.